<a href="https://colab.research.google.com/github/MarcelinaBytes/readmission-prediction-ML/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

raw_url = "https://raw.githubusercontent.com/MarcelinaBytes/readmission-prediction-ML/refs/heads/main/data/raw/diabetic_data.csv"
df = pd.read_csv(raw_url)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# Normalize column names: strip whitespace and lowercase
df.columns = df.columns.map(lambda c: c.strip().lower())

#Confirm dataset loaded correctly

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

#Replace ? with NaN

In [6]:
import numpy as np

df = df.replace("?", np.nan)

#Visualize missingness (top 15 columns)

In [7]:
df.isna().sum().sort_values(ascending=False).head(15)

,0
weight,98569
max_glu_serum,96420
a1cresult,84748
medical_specialty,49949
payer_code,40256
race,2273
diag_3,1423
diag_2,358
diag_1,21
patient_nbr,0


#Drop extremely sparse columns

In [8]:
# Normalize column names: strip spaces and lower-case
df.columns = df.columns.str.strip().str.lower()

# Try drop again with normalized names
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df.shape

(101766, 47)

In [9]:
# Show a few columns
print(df.shape)
print(df.columns[:15].tolist())  # peek

(101766, 47)
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient']


In [10]:
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
to_drop = [c for c in cols_to_drop if c in df.columns]
print("Dropping:", to_drop)

df = df.drop(columns=to_drop, errors='ignore')
print(df.shape)

Dropping: []
(101766, 47)


Continue Pipeline- target transform

In [11]:
# Confirm the target column name after normalization
assert 'readmitted' in df.columns, "Expected 'readmitted' column not found."

# Binary target: 1 = <30, else 0
df['readmitted_flag'] = (df['readmitted'] == '<30').astype(int)
df['readmitted_flag'].value_counts()

,count
readmitted_flag,
0,90409
1,11357


In [12]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/clean_step1.csv', index=False)

In [13]:
check_cols = ['weight', 'payer_code', 'medical_specialty']
present = [c for c in check_cols if c in df.columns]
missing  = [c for c in check_cols if c not in df.columns]
print("Present now:", present)
print("Already absent:", missing)
print("Current shape:", df.shape)

Present now: []
Already absent: ['weight', 'payer_code', 'medical_specialty']
Current shape: (101766, 48)


Feature Engineering

In [14]:
#reload the partially cleaned file

import pandas as pd
df = pd.read_csv('/data/processed/clean_step1.csv')
print(df.shape)
df.head(2)

(101766, 48)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesmed,readmitted,readmitted_flag
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,41,...,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,59,...,Up,No,No,No,No,No,Ch,Yes,>30,0


In [15]:
#create clinically meaningful features

import numpy as np

# --- Utilization aggregates
df['prior_utilization'] = df[['number_outpatient','number_emergency','number_inpatient']].sum(axis=1)

# --- Medication flags
df['high_med_burden'] = (df['num_medications'] >= 10).astype(int)

# --- Insulin usage binary
if 'insulin' in df.columns:
    df['insulin_flag'] = (df['insulin'].str.lower() != 'no').astype(int)

# --- HbA1c categorical to ordered score
if 'a1cresult' in df.columns:
    a1c_map = {'None': 0, 'Normal': 1, '>7': 2, '>8': 3, 'none':0, 'normal':1}
    df['a1c_score'] = df['a1cresult'].map(a1c_map).fillna(0).astype(int)

In [16]:
#map the ICD-9 codes to high level chapters

def icd9_group(code):
    """
    Map ICD-9 code (string) into broad chapter.
    Handles 'V'/'E' codes and numeric ranges.
    Returns one of: circulatory, respiratory, endocrine, injury, neoplasms, digestive, genitourinary, musculoskeletal, mental, symptoms, other, unknown
    """
    if pd.isna(code):
        return 'unknown'
    s = str(code).strip()
    # Non-numeric categories first
    if s.startswith(('V','v')):
        return 'other'
    if s.startswith(('E','e')):
        return 'injury'
    # Try numeric
    try:
        x = float(s)
    except:
        return 'unknown'
    # Ranges (coarse)
    if 390 <= x <= 459 or x == 785:   return 'circulatory'
    if 460 <= x <= 519 or x == 786:   return 'respiratory'
    if 240 <= x <= 279:               return 'endocrine'
    if 800 <= x <= 999:               return 'injury'
    if 140 <= x <= 239:               return 'neoplasms'
    if 520 <= x <= 579 or x == 787:   return 'digestive'
    if 580 <= x <= 629 or x == 788:   return 'genitourinary'
    if 710 <= x <= 739:               return 'musculoskeletal'
    if 290 <= x <= 319:               return 'mental'
    if 780 <= x <= 799:               return 'symptoms'
    return 'other'

for col in ['diag_1','diag_2','diag_3']:
    if col in df.columns:
        df[col + '_group'] = df[col].apply(icd9_group)

# counts across the three diagnosis positions
diag_group_cols = [c for c in df.columns if c.endswith('_group')]
if diag_group_cols:
    counts = (df[diag_group_cols]
              .apply(lambda row: pd.Series(row.values).value_counts(), axis=1)
              .fillna(0))
    counts.columns = [f'diag_count_{c}' for c in counts.columns]
    df = pd.concat([df, counts], axis=1)

In [17]:
#Select/encode features for modeling

from sklearn.preprocessing import OneHotEncoder

# Identify columns
numeric_cols = [
    'time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_outpatient','number_emergency','number_inpatient',
    'prior_utilization','a1c_score'
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

categorical_cols = []
categorical_cols += [c for c in ['gender','age'] if c in df.columns]
categorical_cols += [c for c in ['insulin'] if c in df.columns]  # only if you want original text, else use insulin_flag
categorical_cols += [c for c in ['admission_type_id','discharge_disposition_id','admission_source_id'] if c in df.columns]
categorical_cols += [c for c in ['diag_1_group','diag_2_group','diag_3_group'] if c in df.columns]

# Drop obvious identifiers/leakage
drop_now = [c for c in ['encounter_id','patient_nbr'] if c in df.columns]

base_cols = numeric_cols + categorical_cols + [c for c in df.columns if c.startswith('diag_count_')] + ['readmitted_flag']
base_cols = [c for c in base_cols if c in df.columns]
df_model = df[base_cols].drop(columns=drop_now, errors='ignore')

# One-hot encode categoricals
cat_cols = [c for c in categorical_cols if c in df_model.columns]
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = enc.fit_transform(df_model[cat_cols]) if cat_cols else None
cat_names = enc.get_feature_names_out(cat_cols) if cat_cols else []

# Numeric
import numpy as np
num_cols_final = [c for c in df_model.columns if c in numeric_cols or c.startswith('diag_count_')]
X_num = df_model[num_cols_final].to_numpy(dtype=float) if num_cols_final else np.empty((len(df_model), 0))

# Combine
if X_cat is not None:
    X = np.hstack([X_num, X_cat])
    feature_names = num_cols_final + cat_names.tolist()
else:
    X = X_num
    feature_names = num_cols_final

y = df_model['readmitted_flag'].to_numpy().astype(int)

X.shape, len(feature_names), np.bincount(y)


((101766, 125), 125, array([90409, 11357]))

In [20]:
#save the modeling matrix

import os, joblib
os.makedirs('data/processed', exist_ok=True)
joblib.dump({
    'X': X,
    'y': y,
    'feature_names': feature_names,
    'encoder': enc,
    'cat_cols': cat_cols,
    'num_cols': num_cols_final
}, 'data/processed/model_ready.pkl')

['data/processed/model_ready.pkl']

In [24]:
import os, joblib

os.makedirs('data/processed', exist_ok=True)

joblib.dump(
    {
        'X': X,
        'y': y,
        'feature_names': feature_names,
        'encoder': enc,
        'cat_cols': cat_cols,
        'num_cols': num_cols_final
    },
    'data/processed/model_ready.pkl'
)

['data/processed/model_ready.pkl']